# 00 - Colab setup + split manifests  (ONE-TIME, ALREADY DONE)
**Do not re-run.** The manifests in `splits/` are agreed with Chethan; re-running would regenerate and recommit them. Kept as a record of how they were made. Every other notebook starts with the BOOTSTRAP cell (see 01).

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Drive (checkpoints and run outputs live here, not in the repo)
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/csm-clip'
os.makedirs(DRIVE_ROOT + '/runs', exist_ok=True)
os.makedirs(DRIVE_ROOT + '/splits_backup', exist_ok=True)
print('Drive ready:', DRIVE_ROOT)

In [ ]:
# 3. Clone the private repo. EDIT GH_USER. The token comes from Colab Secrets and is never printed or stored in the notebook.
from google.colab import userdata
GH_USER = 'YOUR_GITHUB_USERNAME'   # <-- edit
token = userdata.get('GITHUB_TOKEN')
%cd /content
if not os.path.exists('/content/csm-clip'):
    !git clone -q https://{token}@github.com/{GH_USER}/csm-clip.git
%cd /content/csm-clip
# strip the token from the saved remote URL
!git remote set-url origin https://github.com/{GH_USER}/csm-clip.git
!git log --oneline | head -5

In [ ]:
# 4. Install dependencies and run the unit tests (CPU tests, should all pass)
!pip install -q -r requirements.txt
!python -m pytest -q

In [ ]:
# 5. Generate the split manifests ONCE (downloads Tiny-GenImage, takes a few minutes)
!python -m src.export_splits --out splits

In [ ]:
# 6. Hashes to send to Chethan, and a Drive backup of the manifests
import hashlib, glob, shutil
for p in sorted(glob.glob('splits/*.json')):
    print(hashlib.sha256(open(p,'rb').read()).hexdigest(), p)
    shutil.copy(p, DRIVE_ROOT + '/splits_backup/')

In [ ]:
# 7. Commit the manifests (set your name/email first)
!git config user.name "YOUR NAME"
!git config user.email "you@example.com"
!git add splits/*.json
!git commit -m "M1: canonical LOGO split manifests (seed 0, per_gen 300)"
# push from Colab with the token, or download splits/*.json and commit from your machine:
# !git push https://{token}@github.com/{GH_USER}/csm-clip.git main